# Silver — tipagem, limpeza, deduplicação e junção

Lê a **bronze** (todas as cargas) e grava a **silver** em `data/silver/`. A silver é uma função
determinística da bronze: rodar duas vezes dá exatamente o mesmo resultado (célula 9 prova).

Tabelas geradas:
- `dim_uf` — UF, código IBGE, região e capital usada como proxy do clima.
- `silver_energia_uf` — consumo EPE por UF. PK: `cod_ibge_uf, data_ref, subsistema, classe, tipo_consumidor`.
- `silver_energia_subsistema` — consumo EPE por região × subsistema. PK: `data_ref, regiao, subsistema, classe, tipo_consumidor`.
- `silver_clima_uf_mes` — clima diário agregado por UF e mês. PK: `cod_ibge_uf, data_ref`.
- `silver_oni_mes` — ONI mensal com fase e intensidade do ENSO. PK: `data_ref`.
- `silver_energia_clima` — energia + clima (código IBGE + ano/mês) + ONI (ano/mês). Mesma PK de `silver_energia_uf`.
- `silver_orfaos_join` — chaves que ficaram sem par no cruzamento.
- `silver_quarentena` — registros que violaram regra de negócio, com a regra e o valor original.

Decisões de projeto:
- Janela 2015–2025 (a EPE vem desde 2004 e o clima só cobre a janela). O ONI **não** é recortado, para permitir defasagens e para a fase do ENSO (episódio de 5 meses) não ser cortada na borda.
- Estorno (consumo negativo) é publicado assim pela EPE: fica na silver com `flag_estorno`, não é descartado.
- Cativo e livre ficam separados na silver; a soma dos dois é feita na Gold.
- Deduplicação: por chave de negócio, vence a versão mais recente (`DataVersao`, depois `_id_carga`). Duplicata exata é colapsada; mesma chave com valores diferentes na mesma versão vai para a quarentena.

In [13]:
import hashlib, json, os, re, unicodedata
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent
DATA = RAIZ / "data"
SILVER = DATA / "silver"
RELATORIOS = SILVER / "_relatorios"

JANELA_INI = pd.Timestamp("2015-01-01")    # meses inclusivos, sempre dia 1
JANELA_FIM = pd.Timestamp("2025-12-01")

LIMIAR_CALOR_C = 35.0       # dia de calor: temperatura máxima >= limiar
LIMIAR_CHUVA_MM = 1.0       # dia de chuva: precipitação >= limiar
FAIXA_TEMP_C = (-30.0, 50.0)
FAIXA_TMAX_C = (-30.0, 55.0)
FAIXA_CHUVA_MM = (0.0, 1000.0)
FAIXA_ONI = (-5.0, 5.0)

## 1. Referências: UF → código IBGE e domínios fechados

Nenhuma fonte traz código IBGE. Ele vem desta dimensão. O cruzamento é por UF (a capital representa o clima do estado).

In [2]:
def slug(texto):
    if not isinstance(texto, str):
        return None
    sem_acento = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "_", sem_acento.lower()).strip("_")


UF_REF = {  # uf: (cod_ibge, nome, regiao, capital)
    "AC": ("12", "Acre", "norte", "Rio Branco"),
    "AL": ("27", "Alagoas", "nordeste", "Maceió"),
    "AP": ("16", "Amapá", "norte", "Macapá"),
    "AM": ("13", "Amazonas", "norte", "Manaus"),
    "BA": ("29", "Bahia", "nordeste", "Salvador"),
    "CE": ("23", "Ceará", "nordeste", "Fortaleza"),
    "DF": ("53", "Distrito Federal", "centro_oeste", "Brasília"),
    "ES": ("32", "Espírito Santo", "sudeste", "Vitória"),
    "GO": ("52", "Goiás", "centro_oeste", "Goiânia"),
    "MA": ("21", "Maranhão", "nordeste", "São Luís"),
    "MT": ("51", "Mato Grosso", "centro_oeste", "Cuiabá"),
    "MS": ("50", "Mato Grosso do Sul", "centro_oeste", "Campo Grande"),
    "MG": ("31", "Minas Gerais", "sudeste", "Belo Horizonte"),
    "PA": ("15", "Pará", "norte", "Belém"),
    "PB": ("25", "Paraíba", "nordeste", "João Pessoa"),
    "PR": ("41", "Paraná", "sul", "Curitiba"),
    "PE": ("26", "Pernambuco", "nordeste", "Recife"),
    "PI": ("22", "Piauí", "nordeste", "Teresina"),
    "RJ": ("33", "Rio de Janeiro", "sudeste", "Rio de Janeiro"),
    "RN": ("24", "Rio Grande do Norte", "nordeste", "Natal"),
    "RS": ("43", "Rio Grande do Sul", "sul", "Porto Alegre"),
    "RO": ("11", "Rondônia", "norte", "Porto Velho"),
    "RR": ("14", "Roraima", "norte", "Boa Vista"),
    "SC": ("42", "Santa Catarina", "sul", "Florianópolis"),
    "SP": ("35", "São Paulo", "sudeste", "São Paulo"),
    "SE": ("28", "Sergipe", "nordeste", "Aracaju"),
    "TO": ("17", "Tocantins", "norte", "Palmas"),
}
dim_uf = pd.DataFrame([(c, uf, n, r, cap) for uf, (c, n, r, cap) in UF_REF.items()],
                      columns=["cod_ibge_uf", "uf", "nome_uf", "regiao", "capital_proxy_clima"])
COD_UF = dict(zip(dim_uf["uf"], dim_uf["cod_ibge_uf"]))
REGIAO_UF = dict(zip(dim_uf["uf"], dim_uf["regiao"]))
UF_POR_COD = dict(zip(dim_uf["cod_ibge_uf"], dim_uf["uf"]))

SUBSISTEMA = {  # "Sistema" da EPE (texto normalizado) -> (sigla, pertence ao SIN)
    "NORDESTE": ("NE", True),
    "NORTE INTERLIGADO": ("N", True),
    "SISTEMAS ISOLADOS": ("ISOLADO", False),
    "SUDESTE / CENTRO - OESTE": ("SE_CO", True),
    "SUL": ("S", True),
}
CLASSES = ["residencial", "comercial", "industrial", "rural", "outros"]
TIPOS = ["cativo", "livre"]
REGIOES = ["norte", "nordeste", "centro_oeste", "sudeste", "sul"]

## 2. Esquemas (tipagem forte) e dicionário de dados

Cada coluna tem tipo e descrição definidos uma vez. `aplicar_schema` converte e recusa tabela com coluna faltando;
`validar_pk` recusa chave nula ou repetida. O dicionário é exportado em CSV (governança).

In [3]:
S, I8, I16, I64, F, D, B = "string", "Int8", "Int16", "Int64", "float64", "datetime64[ns]", "boolean"

COLUNAS = {  # nome: (tipo, descrição)
    "cod_ibge_uf": (S, "Código IBGE da UF (2 dígitos, texto); chave de junção"),
    "uf": (S, "Sigla da UF"),
    "nome_uf": (S, "Nome da UF"),
    "regiao": (S, "Região geográfica: norte, nordeste, centro_oeste, sudeste, sul"),
    "capital_proxy_clima": (S, "Capital cujo clima representa a UF"),
    "data_ref": (D, "Mês de referência (sempre dia 1); chave de junção"),
    "ano": (I16, "Ano de data_ref"),
    "mes": (I8, "Mês de data_ref (1-12)"),
    "subsistema": (S, "Subsistema elétrico: N, NE, SE_CO, S ou ISOLADO"),
    "em_sin": (B, "True se pertence ao SIN; False = sistemas isolados"),
    "classe": (S, "Classe de consumo: residencial, comercial, industrial, rural, outros"),
    "tipo_consumidor": (S, "cativo ou livre"),
    "consumo_mwh": (F, "Consumo faturado no mês, na unidade da EPE (MWh; confirmar no dicionário da EPE). Negativo = estorno"),
    "n_consumidores": (I64, "Número de consumidores"),
    "flag_estorno": (B, "True quando consumo_mwh < 0 (estorno do mercado livre, mantido como publicado)"),
    "data_versao_epe": (D, "Data da versão publicada pela EPE"),
    "temp_media_c": (F, "Média mensal da temperatura média diária (°C)"),
    "temp_max_media_c": (F, "Média mensal da temperatura máxima diária (°C)"),
    "temp_max_abs_c": (F, "Maior temperatura máxima do mês (°C)"),
    "precip_total_mm": (F, "Precipitação total do mês (mm)"),
    "dias_chuva": (I16, "Dias com precipitação >= LIMIAR_CHUVA_MM"),
    "dias_calor": (I16, "Dias com temperatura máxima >= LIMIAR_CALOR_C"),
    "dias_com_dado": (I16, "Dias válidos no mês"),
    "dias_no_mes": (I8, "Dias do mês no calendário"),
    "cobertura_completa": (B, "True se todos os dias do mês têm dado válido"),
    "clima_cobertura_completa": (B, "cobertura_completa do clima do mês (nulo se sem clima)"),
    "lat_grade": (F, "Latitude da célula de grade ERA5 usada pela API"),
    "lon_grade": (F, "Longitude da célula de grade ERA5 usada pela API"),
    "oni": (F, "Oceanic Niño Index (média móvel centrada de 3 meses). Em modelos, usar defasagem >= 1 mês"),
    "fase_enso": (S, "el_nino, la_nina ou neutro (episódio: |ONI| >= 0,5 por 5 meses seguidos, critério NOAA)"),
    "intensidade_enso": (S, "fraca, moderada, forte ou muito_forte (nulo se neutro)"),
    "flag_sem_clima": (B, "True se a UF-mês não tem clima (órfão do join)"),
    "flag_sem_oni": (B, "True se o mês não tem ONI (órfão do join)"),
    "lado": (S, "Lado órfão: energia_sem_clima ou clima_sem_energia"),
    "motivo": (S, "Motivo do órfão"),
    "tabela": (S, "Tabela silver de origem do registro rejeitado"),
    "regra": (S, "Regra de negócio violada"),
    "valor_original": (S, "Valor original (ou chave) que violou a regra"),
    "_id_carga": (S, "Carga da bronze que originou o registro (a mais recente da chave)"),
    "_id_carga_clima": (S, "Carga da bronze do clima usada no join"),
    "_id_carga_oni": (S, "Carga da bronze do ONI usada no join"),
    "_hash_registro": (S, "Hash do registro bruto na bronze"),
}

PK_ENERGIA = ["cod_ibge_uf", "data_ref", "subsistema", "classe", "tipo_consumidor"]
COLS_ENERGIA = ["cod_ibge_uf", "uf", "data_ref", "ano", "mes", "regiao", "subsistema", "em_sin", "classe",
                "tipo_consumidor", "consumo_mwh", "n_consumidores", "flag_estorno", "data_versao_epe",
                "_id_carga", "_hash_registro"]

SCHEMAS = {
    "dim_uf": {"pk": ["cod_ibge_uf"],
               "colunas": ["cod_ibge_uf", "uf", "nome_uf", "regiao", "capital_proxy_clima"]},
    "silver_energia_uf": {"pk": PK_ENERGIA, "colunas": COLS_ENERGIA},
    "silver_energia_subsistema": {
        "pk": ["data_ref", "regiao", "subsistema", "classe", "tipo_consumidor"],
        "colunas": ["data_ref", "ano", "mes", "regiao", "subsistema", "em_sin", "classe", "tipo_consumidor",
                    "consumo_mwh", "n_consumidores", "flag_estorno", "data_versao_epe", "_id_carga", "_hash_registro"]},
    "silver_clima_uf_mes": {
        "pk": ["cod_ibge_uf", "data_ref"],
        "colunas": ["cod_ibge_uf", "uf", "data_ref", "ano", "mes", "temp_media_c", "temp_max_media_c",
                    "temp_max_abs_c", "precip_total_mm", "dias_chuva", "dias_calor", "dias_com_dado",
                    "dias_no_mes", "cobertura_completa", "lat_grade", "lon_grade", "_id_carga"]},
    "silver_oni_mes": {
        "pk": ["data_ref"],
        "colunas": ["data_ref", "ano", "mes", "oni", "fase_enso", "intensidade_enso", "_id_carga"]},
    "silver_energia_clima": {
        "pk": PK_ENERGIA,
        "colunas": COLS_ENERGIA + ["temp_media_c", "temp_max_media_c", "temp_max_abs_c", "precip_total_mm",
                                   "dias_chuva", "dias_calor", "clima_cobertura_completa", "oni", "fase_enso",
                                   "intensidade_enso", "flag_sem_clima", "flag_sem_oni",
                                   "_id_carga_clima", "_id_carga_oni"]},
    "silver_orfaos_join": {"pk": ["lado", "cod_ibge_uf", "data_ref"],
                           "colunas": ["lado", "cod_ibge_uf", "uf", "data_ref", "motivo"]},
    "silver_quarentena": {"pk": [],
                          "colunas": ["tabela", "regra", "valor_original", "_id_carga", "_hash_registro"]},
}

VALIDACOES, VERBOSO = [], True


def checar(nome, ok, detalhe="", critico=True):
    ok = bool(ok)
    VALIDACOES.append({"validacao": nome, "ok": ok, "detalhe": str(detalhe), "critico": critico})
    if VERBOSO or not ok:
        print(f"{'OK    ' if ok else 'FALHA '} {nome}" + (f" | {detalhe}" if detalhe else ""))
    if critico:
        assert ok, f"validação crítica falhou: {nome} {detalhe}"


def aplicar_schema(df, tabela):
    nomes = SCHEMAS[tabela]["colunas"]
    faltam = [c for c in nomes if c not in df.columns]
    assert not faltam, f"{tabela}: colunas ausentes {faltam}"
    out = df[nomes].copy()
    for c in nomes:
        out[c] = out[c].astype(COLUNAS[c][0])
    return out


def validar_pk(df, tabela):
    pk = SCHEMAS[tabela]["pk"]
    if not pk:
        return
    checar(f"{tabela}: PK sem nulos", not df[pk].isna().any().any())
    repetidas = int(df.duplicated(pk).sum())
    checar(f"{tabela}: PK única ({', '.join(pk)})", repetidas == 0, f"{repetidas} repetidas, {len(df)} linhas")


def hash_tabela(df, ignorar_linhagem=False):
    if ignorar_linhagem:
        df = df.drop(columns=[c for c in df.columns if c.startswith("_")])
    h = pd.util.hash_pandas_object(df.reset_index(drop=True), index=False).values
    return hashlib.sha256(h.tobytes() + ",".join(df.columns).encode()).hexdigest()[:16]


def dicionario():
    linhas = [{"tabela": t, "coluna": c, "tipo": COLUNAS[c][0], "chave_primaria": c in s["pk"],
               "descricao": COLUNAS[c][1]} for t, s in SCHEMAS.items() for c in s["colunas"]]
    return pd.DataFrame(linhas)

## 3. Funções comuns: leitura, quarentena e deduplicação

- `ler_bronze` junta **todas** as cargas.
- `separar` aplica as regras de negócio em ordem; cada linha cai só na primeira regra que violar.
- `deduplicar` mantém a versão mais recente por chave, colapsa duplicata exata e põe conflito na quarentena.

In [4]:
def ler_bronze(fonte):
    return pd.read_parquet(DATA / "bronze" / fonte).astype("string")


def separar(d, b, regras, tabela, q):
    invalido = pd.Series(False, index=d.index)
    resumo = {}
    for regra, mascara, coluna_bruta in regras:
        novo = mascara & ~invalido
        if novo.any():
            q.append(pd.DataFrame({
                "tabela": tabela, "regra": regra,
                "valor_original": b.loc[novo.index[novo], coluna_bruta].astype("string").fillna("<nulo>").astype(str).values,
                "_id_carga": d.loc[novo, "_id_carga"].astype(str).values,
                "_hash_registro": d.loc[novo, "_hash_registro"].astype(str).values}))
            resumo[regra] = int(novo.sum())
        invalido |= novo
    return d[~invalido], resumo


def deduplicar(d, pk, valores, tabela, q, st):
    mais_nova = d.groupby(pk, dropna=False)["_versao"].transform("max")
    ultimo = d[d["_versao"] == mais_nova]
    st["versoes_antigas_descartadas"] = len(d) - len(ultimo)

    unico = ultimo.drop_duplicates(subset=pk + valores)
    st["duplicatas_exatas_descartadas"] = len(ultimo) - len(unico)

    conflito = unico.duplicated(subset=pk, keep=False)
    st["conflitos_para_quarentena"] = int(conflito.sum())
    if conflito.any():
        c = unico[conflito]
        q.append(pd.DataFrame({
            "tabela": tabela, "regra": "chave_duplicada_valores_conflitantes",
            "valor_original": c[pk].astype(str).agg("|".join, axis=1).values,
            "_id_carga": c["_id_carga"].astype(str).values,
            "_hash_registro": c["_hash_registro"].astype(str).values}))
    return unico[~conflito]

## 4. EPE — consumo por UF e por subsistema

Regras: data `AAAAMMDD` válida; consumo numérico (negativo é aceito e sinalizado); consumidores inteiro >= 0;
classe, tipo, sistema, região e UF dentro do domínio. Padronização: classe/tipo/região em minúsculas sem acento;
`Sistema` vira sigla de subsistema + `em_sin`.

In [5]:
def tratar_epe(fonte, q, rel):
    por_uf = fonte.endswith("_uf")
    tabela = "silver_" + fonte.replace("epe_consumo_", "energia_")
    b = ler_bronze(fonte)
    st = {"linhas_bronze": len(b)}

    d = pd.DataFrame({"_id_carga": b["_id_carga"], "_hash_registro": b["_hash_registro"]})
    d["data_ref"] = (pd.to_datetime(b["Data"].str.strip(), format="%Y%m%d", errors="coerce")
                       .dt.to_period("M").dt.to_timestamp())
    d["classe"] = b["Classe"].astype(object).map(slug)
    d["tipo_consumidor"] = b["TipoConsumidor"].astype(object).map(slug)
    d["regiao_epe"] = b["Regiao"].astype(object).map(slug)
    sis = b["Sistema"].astype(object).map(
        lambda s: re.sub(r"\s+", " ", s.strip().upper()) if isinstance(s, str) else None)
    d["subsistema"] = sis.map(lambda s: SUBSISTEMA.get(s, (None, None))[0])
    d["em_sin"] = sis.map(lambda s: SUBSISTEMA.get(s, (None, None))[1])
    d["consumo_mwh"] = pd.to_numeric(b["Consumo"], errors="coerce").astype("float64")
    d["n_consumidores"] = pd.to_numeric(b["Consumidores"], errors="coerce").astype("float64")
    d["data_versao_epe"] = pd.to_datetime(b["DataVersao"], errors="coerce")
    d["_versao"] = b["DataVersao"].fillna("").astype(str) + "|" + b["_id_carga"].astype(str)
    if por_uf:
        d["uf"] = b["UF"].astype(object).map(lambda s: s.strip().upper() if isinstance(s, str) else None)

    fora = d["data_ref"].notna() & ~d["data_ref"].between(JANELA_INI, JANELA_FIM)
    st["fora_da_janela"] = int(fora.sum())
    d = d[~fora]

    regras = [
        ("data_invalida", d["data_ref"].isna(), "Data"),
        ("consumo_nao_numerico", d["consumo_mwh"].isna(), "Consumo"),
        ("consumidores_invalido", d["n_consumidores"].isna() | (d["n_consumidores"] < 0)
                                  | (d["n_consumidores"] % 1 != 0), "Consumidores"),
        ("classe_fora_do_dominio", ~d["classe"].isin(CLASSES), "Classe"),
        ("tipo_fora_do_dominio", ~d["tipo_consumidor"].isin(TIPOS), "TipoConsumidor"),
        ("sistema_fora_do_dominio", d["subsistema"].isna(), "Sistema"),
        ("regiao_fora_do_dominio", ~d["regiao_epe"].isin(REGIOES), "Regiao"),
    ]
    if por_uf:
        regras.append(("uf_fora_do_dominio", ~d["uf"].isin(list(COD_UF)), "UF"))
    d, st["quarentena_por_regra"] = separar(d, b, regras, tabela, q)

    pk = (["uf"] if por_uf else ["regiao_epe"]) + ["data_ref", "subsistema", "classe", "tipo_consumidor"]
    d = deduplicar(d, pk, ["consumo_mwh", "n_consumidores"], tabela, q, st).copy()

    d["ano"] = d["data_ref"].dt.year
    d["mes"] = d["data_ref"].dt.month
    d["flag_estorno"] = d["consumo_mwh"] < 0
    if por_uf:
        d["cod_ibge_uf"] = d["uf"].map(COD_UF)
        d["regiao"] = d["uf"].map(REGIAO_UF)
        divergentes = int((d["regiao"] != d["regiao_epe"]).sum())
        checar(f"{tabela}: região da EPE coincide com a da UF", divergentes == 0,
               f"{divergentes} divergentes", critico=False)
        n_sub = d.groupby("uf")["subsistema"].nunique()
        st["ufs_com_mais_de_um_subsistema"] = {k: int(v) for k, v in n_sub[n_sub > 1].items()}
    else:
        d["regiao"] = d["regiao_epe"]

    out = aplicar_schema(d.sort_values(SCHEMAS[tabela]["pk"]), tabela).reset_index(drop=True)
    validar_pk(out, tabela)
    st["linhas_silver"] = len(out)
    st["estornos_mantidos"] = int(out["flag_estorno"].sum())
    st["consumo_total_mwh"] = float(out["consumo_mwh"].sum())
    rel[tabela] = st
    return out

## 5. Open-Meteo — clima diário → mensal por UF

A UF vem do nome do arquivo raw (`_arquivo_raw`). Regras por dia: UF válida, data válida, nenhuma variável nula,
faixas físicas, temperatura média <= máxima, precipitação >= 0. Unidades conferidas no sidecar. Agregação mensal:
médias de temperatura, máxima absoluta, chuva total, dias de chuva, dias de calor e cobertura do mês.

In [6]:
def checar_unidades_clima():
    carga = str(pd.read_parquet(DATA / "bronze" / "openmeteo_clima", columns=["_id_carga"])["_id_carga"].max())
    metas = sorted((DATA / "raw" / "openmeteo_clima" / carga).glob("*.meta.json"))
    checar("clima: sidecars da carga mais recente encontrados", len(metas) > 0, f"{len(metas)} em {carga}")
    unidades = {json.dumps(json.loads(p.read_text(encoding="utf-8")).get("_unidades"), sort_keys=True, ensure_ascii=False)
                for p in metas}
    u = json.loads(next(iter(unidades)))
    checar("clima: unidades uniformes (°C e mm)",
           len(unidades) == 1 and u.get("temperature_2m_mean") == "°C"
           and u.get("temperature_2m_max") == "°C" and u.get("precipitation_sum") == "mm", u)


def tratar_clima(q, rel):
    tabela = "silver_clima_uf_mes"
    b = ler_bronze("openmeteo_clima")
    st = {"linhas_bronze": len(b)}

    d = pd.DataFrame({"_id_carga": b["_id_carga"], "_hash_registro": b["_hash_registro"]})
    d["uf"] = b["_arquivo_raw"].str.extract(r"clima_([A-Z]{2})_\d{4}\.json")[0]
    d["data"] = pd.to_datetime(b["time"], format="%Y-%m-%d", errors="coerce")
    for novo, velho in (("t_media", "temperature_2m_mean"), ("t_max", "temperature_2m_max"),
                        ("chuva", "precipitation_sum"), ("lat", "latitude"), ("lon", "longitude")):
        d[novo] = pd.to_numeric(b[velho], errors="coerce").astype("float64")
    d["_versao"] = b["_id_carga"].astype(str)

    fim_dia = JANELA_FIM + pd.offsets.MonthEnd(0)
    fora = d["data"].notna() & ((d["data"] < JANELA_INI) | (d["data"] > fim_dia))
    st["fora_da_janela"] = int(fora.sum())
    d = d[~fora]

    regras = [
        ("uf_nao_identificada", ~d["uf"].isin(list(COD_UF)), "_arquivo_raw"),
        ("data_invalida", d["data"].isna(), "time"),
        ("variavel_nula", d[["t_media", "t_max", "chuva"]].isna().any(axis=1), "temperature_2m_mean"),
        ("coordenada_ausente", d[["lat", "lon"]].isna().any(axis=1), "latitude"),
        ("temp_media_fora_da_faixa", ~d["t_media"].between(*FAIXA_TEMP_C), "temperature_2m_mean"),
        ("temp_max_fora_da_faixa", ~d["t_max"].between(*FAIXA_TMAX_C), "temperature_2m_max"),
        ("temp_media_maior_que_max", d["t_media"] > d["t_max"] + 0.05, "temperature_2m_mean"),
        ("chuva_fora_da_faixa", ~d["chuva"].between(*FAIXA_CHUVA_MM), "precipitation_sum"),
    ]
    d, st["quarentena_por_regra"] = separar(d, b, regras, tabela, q)
    d = deduplicar(d, ["uf", "data"], ["t_media", "t_max", "chuva", "lat", "lon"], tabela, q, st).copy()

    celulas = d.groupby("uf")[["lat", "lon"]].nunique().max().max()
    checar("clima: uma célula de grade por UF", celulas == 1, f"máx {celulas}")

    d["data_ref"] = d["data"].dt.to_period("M").dt.to_timestamp()
    d["_calor"] = d["t_max"] >= LIMIAR_CALOR_C
    d["_chuva"] = d["chuva"] >= LIMIAR_CHUVA_MM
    d = d.sort_values(["uf", "data"])
    m = d.groupby(["uf", "data_ref"], as_index=False).agg(
        temp_media_c=("t_media", "mean"), temp_max_media_c=("t_max", "mean"), temp_max_abs_c=("t_max", "max"),
        precip_total_mm=("chuva", "sum"), dias_chuva=("_chuva", "sum"), dias_calor=("_calor", "sum"),
        dias_com_dado=("data", "size"), lat_grade=("lat", "first"), lon_grade=("lon", "first"),
        _id_carga=("_id_carga", "max"))
    m["cod_ibge_uf"] = m["uf"].map(COD_UF)
    m["ano"] = m["data_ref"].dt.year
    m["mes"] = m["data_ref"].dt.month
    m["dias_no_mes"] = m["data_ref"].dt.days_in_month
    m["cobertura_completa"] = m["dias_com_dado"] == m["dias_no_mes"]

    out = aplicar_schema(m.sort_values(SCHEMAS[tabela]["pk"]), tabela).reset_index(drop=True)
    validar_pk(out, tabela)
    st["linhas_silver"] = len(out)
    st["meses_com_cobertura_parcial"] = int((~out["cobertura_completa"]).sum())
    rel[tabela] = st
    return out

## 6. NOAA ONI — largo → longo, sentinela e fase do ENSO

O sentinela é lido do rodapé gravado no sidecar (sem número fixo no código). A fase segue o critério da NOAA:
El Niño = ONI >= +0,5 por pelo menos 5 meses seguidos; La Niña = ONI <= -0,5 pelo mesmo período; senão neutro.
A intensidade usa |ONI|: < 1,0 fraca, < 1,5 moderada, < 2,0 forte, >= 2,0 muito forte.

In [7]:
def sentinela_oni(carga):
    meta = json.loads((DATA / "raw" / "noaa_oni" / carga / "oni.data.meta.json").read_text(encoding="utf-8"))
    achados = re.findall(r"-\d{2,}\.\d+", " ".join(meta["_rodape"]))
    assert achados, f"sentinela não encontrado no rodapé: {meta['_rodape']}"
    return float(achados[0])


def fase_enso(oni):
    fase = pd.Series("neutro", index=oni.index, dtype="object")
    for nome, cond in (("el_nino", oni >= 0.5), ("la_nina", oni <= -0.5)):
        bloco = (cond != cond.shift()).cumsum()
        tamanho = cond.groupby(bloco).transform("size")
        fase[cond & (tamanho >= 5)] = nome
    fase[oni.isna()] = None
    return fase


def intensidade_enso(oni, fase):
    a = oni.abs()
    nivel = pd.Series(np.select([a >= 2.0, a >= 1.5, a >= 1.0], ["muito_forte", "forte", "moderada"], "fraca"),
                      index=oni.index, dtype="object")
    nivel[fase.isna() | (fase == "neutro")] = None
    return nivel


def tratar_oni(q, rel):
    tabela = "silver_oni_mes"
    b = ler_bronze("noaa_oni")
    st = {"linhas_bronze": len(b)}
    sentinela = sentinela_oni(str(b["_id_carga"].max()))
    st["sentinela"] = sentinela
    meses = [f"m{i:02d}" for i in range(1, 13)]

    d = b[["_id_carga", "_hash_registro"] + meses].copy()
    d["ano"] = pd.to_numeric(b["ano"], errors="coerce").astype("float64")
    d["_versao"] = b["_id_carga"].astype(str)
    d, st["quarentena_por_regra"] = separar(d, b, [("ano_invalido", d["ano"].isna(), "ano")], tabela, q)
    d = deduplicar(d, ["ano"], meses, tabela, q, st)

    longo = d.melt(id_vars=["ano", "_id_carga", "_hash_registro"], value_vars=meses,
                   var_name="mes_col", value_name="txt")
    longo["mes"] = longo["mes_col"].astype(str).str[1:].astype(int)
    longo["oni"] = pd.to_numeric(longo["txt"], errors="coerce").astype("float64")

    sem_dado = longo["oni"] == sentinela
    st["meses_sentinela_descartados"] = int(sem_dado.sum())
    longo = longo[~sem_dado]

    ruim = longo["oni"].isna() | ~longo["oni"].between(*FAIXA_ONI)
    if ruim.any():
        r = longo[ruim]
        q.append(pd.DataFrame({"tabela": tabela, "regra": "oni_invalido_ou_fora_da_faixa",
                               "valor_original": (r["mes_col"].astype(str) + "=" + r["txt"].astype(str)).values,
                               "_id_carga": r["_id_carga"].astype(str).values,
                               "_hash_registro": r["_hash_registro"].astype(str).values}))
    st["oni_invalido"] = int(ruim.sum())
    longo = longo[~ruim].copy()

    longo["data_ref"] = pd.to_datetime({"year": longo["ano"].astype(int), "month": longo["mes"], "day": 1})
    longo = longo.sort_values("data_ref")
    completa = pd.date_range(longo["data_ref"].min(), longo["data_ref"].max(), freq="MS")
    checar("ONI: série mensal sem lacunas", len(completa) == len(longo),
           f"{len(completa) - len(longo)} meses faltando", critico=False)
    serie = longo.set_index("data_ref")["oni"].reindex(completa)
    fase = fase_enso(serie)
    nivel = intensidade_enso(serie, fase)
    longo["fase_enso"] = longo["data_ref"].map(fase)
    longo["intensidade_enso"] = longo["data_ref"].map(nivel)

    out = aplicar_schema(longo, tabela).sort_values("data_ref").reset_index(drop=True)
    validar_pk(out, tabela)
    na_janela = out[out["data_ref"].between(JANELA_INI, JANELA_FIM)]
    st["linhas_silver"] = len(out)
    st["meses_na_janela_por_fase"] = {k: int(v) for k, v in na_janela["fase_enso"].value_counts().items()}
    rel[tabela] = st
    return out

## 7. Junção energia × clima (código IBGE + ano/mês) × ONI (ano/mês)

`left join` a partir da energia, com `validate="m:1"` (clima e ONI têm uma linha por chave). Órfãos:
UF-mês da energia sem clima, UF-mês do clima sem energia, mês da energia sem ONI. A energia nunca é descartada pelo join:
a linha fica com `flag_sem_clima` / `flag_sem_oni`. O relatório compara linhas e consumo antes e depois.

In [8]:
def juntar(energia, clima, oni, rel):
    chaves = ["cod_ibge_uf", "data_ref"]
    ke = energia[chaves].drop_duplicates()
    kc = clima[chaves]
    cruz = ke.merge(kc, on=chaves, how="outer", indicator=True)
    so_energia = cruz[cruz["_merge"].astype(str) == "left_only"]
    so_clima = cruz[cruz["_merge"].astype(str) == "right_only"]

    cols_clima = ["temp_media_c", "temp_max_media_c", "temp_max_abs_c", "precip_total_mm",
                  "dias_chuva", "dias_calor", "cobertura_completa", "_id_carga"]
    c = clima[chaves + cols_clima].rename(columns={"cobertura_completa": "clima_cobertura_completa",
                                                   "_id_carga": "_id_carga_clima"})
    j = energia.merge(c, on=chaves, how="left", validate="m:1", indicator="_m_clima")
    j["flag_sem_clima"] = j["_m_clima"].astype(str) == "left_only"

    o = oni[["data_ref", "oni", "fase_enso", "intensidade_enso", "_id_carga"]].rename(
        columns={"_id_carga": "_id_carga_oni"})
    j = j.merge(o, on="data_ref", how="left", validate="m:1", indicator="_m_oni")
    j["flag_sem_oni"] = j["_m_oni"].astype(str) == "left_only"
    j = j.drop(columns=["_m_clima", "_m_oni"])

    checar("join: nº de linhas da energia preservado", len(j) == len(energia), f"{len(energia)} -> {len(j)}")
    checar("join: consumo total preservado", np.isclose(j["consumo_mwh"].sum(), energia["consumo_mwh"].sum()))

    orfaos = pd.concat([
        so_energia.assign(lado="energia_sem_clima", motivo="UF-mês com consumo e sem clima"),
        so_clima.assign(lado="clima_sem_energia", motivo="UF-mês com clima e sem consumo")])
    orfaos["uf"] = orfaos["cod_ibge_uf"].map(UF_POR_COD)
    orfaos = aplicar_schema(orfaos, "silver_orfaos_join").sort_values(
        SCHEMAS["silver_orfaos_join"]["pk"]).reset_index(drop=True)
    validar_pk(orfaos, "silver_orfaos_join")

    out = aplicar_schema(j.sort_values(PK_ENERGIA), "silver_energia_clima").reset_index(drop=True)
    validar_pk(out, "silver_energia_clima")

    total = out["consumo_mwh"].sum()
    sem_clima = out["flag_sem_clima"]
    rel["join"] = {
        "linhas_energia_antes": len(energia), "linhas_apos_join": len(out),
        "uf_meses_energia": len(ke), "uf_meses_clima": len(kc),
        "uf_meses_energia_sem_clima": len(so_energia), "uf_meses_clima_sem_energia": len(so_clima),
        "linhas_sem_clima": int(sem_clima.sum()),
        "pct_linhas_sem_clima": round(100 * sem_clima.mean(), 4),
        "mwh_sem_clima": float(out.loc[sem_clima, "consumo_mwh"].sum()),
        "pct_mwh_sem_clima": round(100 * float(out.loc[sem_clima, "consumo_mwh"].sum()) / total, 4),
        "linhas_sem_oni": int(out["flag_sem_oni"].sum()),
        "meses_energia_sem_oni": sorted(out.loc[out["flag_sem_oni"], "data_ref"].dt.strftime("%Y-%m").unique().tolist()),
        "linhas_com_clima_de_cobertura_parcial": int((~out["clima_cobertura_completa"].fillna(True)).sum()),
        "consumo_total_mwh": float(total),
    }
    return out, orfaos

## 8. Orquestração: construir, gravar e relatar

`construir_silver` não grava nada; `gravar` escreve cada tabela por substituição atômica (arquivo temporário + `os.replace`)
em caminho fixo, sem id de execução. Nenhuma coluna das tabelas guarda a hora da execução (ela fica só no relatório),
por isso rodar de novo reproduz os mesmos bytes lógicos.

In [9]:
def construir_silver(verboso=True):
    global VERBOSO
    VERBOSO = verboso
    VALIDACOES.clear()
    q, rel = [], {"janela": [str(JANELA_INI.date()), str(JANELA_FIM.date())]}

    checar_unidades_clima()
    dim = aplicar_schema(dim_uf, "dim_uf")
    validar_pk(dim, "dim_uf")

    energia = tratar_epe("epe_consumo_uf", q, rel)
    subsistema = tratar_epe("epe_consumo_subsistema", q, rel)
    checar("dim_uf cobre exatamente as UFs da EPE", set(energia["uf"]) == set(dim["uf"]),
           f"{energia['uf'].nunique()} UFs na EPE")
    clima = tratar_clima(q, rel)
    oni = tratar_oni(q, rel)
    unida, orfaos = juntar(energia, clima, oni, rel)

    colunas_q = SCHEMAS["silver_quarentena"]["colunas"]
    quar = pd.concat(q, ignore_index=True) if q else pd.DataFrame(columns=colunas_q)
    quar = (quar.sort_values("_id_carga")          # a mesma falha vista em várias cargas conta uma vez (a mais recente)
                .drop_duplicates(["tabela", "regra", "valor_original", "_hash_registro"], keep="last"))
    quar = aplicar_schema(quar.sort_values(colunas_q), "silver_quarentena").reset_index(drop=True)
    rel["quarentena_por_tabela_e_regra"] = {f"{t}/{r}": int(n) for (t, r), n in
                                            quar.groupby(["tabela", "regra"]).size().items()}

    tabelas = {"dim_uf": dim, "silver_energia_uf": energia, "silver_energia_subsistema": subsistema,
               "silver_clima_uf_mes": clima, "silver_oni_mes": oni, "silver_energia_clima": unida,
               "silver_orfaos_join": orfaos, "silver_quarentena": quar}
    rel["linhas_por_tabela"] = {k: len(v) for k, v in tabelas.items()}
    rel["validacoes"] = list(VALIDACOES)
    return tabelas, rel


def gravar(tabelas, rel):
    SILVER.mkdir(parents=True, exist_ok=True)
    RELATORIOS.mkdir(parents=True, exist_ok=True)
    for nome, df in tabelas.items():
        tmp = SILVER / f"{nome}.parquet.tmp"
        df.to_parquet(tmp, index=False)
        os.replace(tmp, SILVER / f"{nome}.parquet")
    dicionario().to_csv(RELATORIOS / "dicionario_silver.csv", index=False, encoding="utf-8")
    rel = {**rel, "gerado_em": datetime.now(timezone.utc).isoformat(timespec="seconds")}
    (RELATORIOS / "silver_relatorio.json").write_text(
        json.dumps(rel, ensure_ascii=False, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)),
        encoding="utf-8")


def executar_silver(verboso=True):
    tabelas, rel = construir_silver(verboso)
    gravar(tabelas, rel)
    return tabelas, rel


def hashes_em_disco(ignorar_linhagem=False):
    return {p.name.removesuffix(".parquet"): hash_tabela(pd.read_parquet(p), ignorar_linhagem)
            for p in sorted(SILVER.glob("*.parquet"))}

## 9. Execução e relatório do cruzamento

In [10]:
tabelas, rel = executar_silver()
H1 = hashes_em_disco()

print("\nlinhas por tabela:", rel["linhas_por_tabela"])
print("\nquarentena:", rel["quarentena_por_tabela_e_regra"] or "vazia")
print("\nimpacto do cruzamento:")
for k, v in rel["join"].items():
    print(f"  {k}: {v}")

OK     clima: sidecars da carga mais recente encontrados | 297 em 20261008T014338Z
OK     clima: unidades uniformes (°C e mm) | {'precipitation_sum': 'mm', 'temperature_2m_max': '°C', 'temperature_2m_mean': '°C', 'time': 'iso8601'}
OK     dim_uf: PK sem nulos
OK     dim_uf: PK única (cod_ibge_uf) | 0 repetidas, 27 linhas
OK     silver_energia_uf: região da EPE coincide com a da UF | 0 divergentes
OK     silver_energia_uf: PK sem nulos
OK     silver_energia_uf: PK única (cod_ibge_uf, data_ref, subsistema, classe, tipo_consumidor) | 0 repetidas, 32875 linhas
OK     silver_energia_subsistema: PK sem nulos
OK     silver_energia_subsistema: PK única (data_ref, regiao, subsistema, classe, tipo_consumidor) | 0 repetidas, 9974 linhas
OK     dim_uf cobre exatamente as UFs da EPE | 27 UFs na EPE
OK     clima: uma célula de grade por UF | máx 1
OK     silver_clima_uf_mes: PK sem nulos
OK     silver_clima_uf_mes: PK única (cod_ibge_uf, data_ref) | 0 repetidas, 3564 linhas
OK     ONI: série mensal 

## 10. Prova de idempotência (dupla execução)

Roda a silver de novo sobre a mesma bronze e compara o hash de conteúdo de cada tabela lida do disco, o número de arquivos
e as linhas. Se a PK, a deduplicação e a escrita estiverem corretas, nada muda e nada duplica.

In [11]:
tabelas2, rel2 = executar_silver(verboso=False)
H2 = hashes_em_disco()
VERBOSO = True

print(f"{'tabela':<28}{'hash 1ª':<18}{'hash 2ª':<18}resultado")
for nome in sorted(H1):
    print(f"{nome:<28}{H1[nome]:<18}{H2.get(nome, '-'):<18}{'igual' if H1[nome] == H2.get(nome) else 'DIFERENTE'}")

checar("idempotência: mesmas tabelas em disco", set(H1) == set(H2))
checar("idempotência: hash idêntico em todas as tabelas", H1 == H2)
checar("idempotência: contagem de linhas idêntica", rel["linhas_por_tabela"] == rel2["linhas_por_tabela"])

tabela                      hash 1ª           hash 2ª           resultado
dim_uf                      7ac6d1bafd683cf6  7ac6d1bafd683cf6  igual
silver_clima_uf_mes         7773df107dc267ed  7773df107dc267ed  igual
silver_energia_clima        6666859a9e95be3c  6666859a9e95be3c  igual
silver_energia_subsistema   2aa18a2a06b39ede  2aa18a2a06b39ede  igual
silver_energia_uf           dc9383b5ffe94c42  dc9383b5ffe94c42  igual
silver_oni_mes              1ab72dfdefc3005f  1ab72dfdefc3005f  igual
silver_orfaos_join          eefd005c891d1073  eefd005c891d1073  igual
silver_quarentena           6feb3e8495a3ef9f  6feb3e8495a3ef9f  igual
OK     idempotência: mesmas tabelas em disco
OK     idempotência: hash idêntico em todas as tabelas
OK     idempotência: contagem de linhas idêntica


Prova extra (opcional): rode a ingestão de novo para criar uma carga nova **sem** mudança nos dados da origem,
depois rode esta célula. O conteúdo (sem as colunas de linhagem `_...`) deve continuar igual.

In [12]:
HC1 = hashes_em_disco(ignorar_linhagem=True)
executar_silver(verboso=False)
HC2 = hashes_em_disco(ignorar_linhagem=True)
VERBOSO = True
checar("reingestão sem mudança: conteúdo idêntico", HC1 == HC2,
       "rode a ingestão de novo antes desta célula; se a EPE publicou nova DataVersao, o conteúdo muda de verdade",
       critico=False)

OK     reingestão sem mudança: conteúdo idêntico | rode a ingestão de novo antes desta célula; se a EPE publicou nova DataVersao, o conteúdo muda de verdade
